In [ ]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.6 MB/s eta 0:00:00


In [1]:
from huggingface_hub import login
login(token="add your keyxxxx")

In [2]:
import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
import torch
from torch.utils.data import DataLoader
import os
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset,DatasetDict

In [5]:
# ds = load_dataset("ronig/pdb_sequences", token=True)
ds = load_dataset("csv", data_files="disease.csv")

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
ds = ds.select_columns(['sequence','disease'])
ds = ds.filter(lambda example: (len(example['sequence']) <= 512) & ( example['disease']=='ebola'))
ds

Filter:   0%|          | 0/20096 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'disease'],
        num_rows: 2868
    })
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path to your saved checkpoint folder
checkpoint_path = 'Delower/Affinity_PPI'

protein_1_seq=
protein_2_seq=

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Load model
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

# Set model to evaluation mode and move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/417 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/61.5M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Llama4ForCausalLM(
  (model): Llama4TextModel(
    (embed_tokens): Embedding(56, 192, padding_idx=0)
    (layers): ModuleList(
      (0-2): 3 x Llama4TextDecoderLayer(
        (self_attn): Llama4TextAttention(
          (q_proj): Linear(in_features=192, out_features=128, bias=False)
          (k_proj): Linear(in_features=192, out_features=128, bias=False)
          (v_proj): Linear(in_features=192, out_features=128, bias=False)
          (o_proj): Linear(in_features=128, out_features=192, bias=False)
          (qk_norm): Llama4TextL2Norm(eps=1e-06)
        )
        (feed_forward): Llama4TextMoe(
          (experts): Llama4TextExperts(
            (act_fn): SiLUActivation()
          )
          (router): Llama4Router(in_features=192, out_features=16, bias=False)
          (shared_expert): Llama4TextMLP(
            (gate_proj): Linear(in_features=192, out_features=512, bias=False)
            (up_proj): Linear(in_features=192, out_features=512, bias=False)
            (down_proj): Lin

# Inference

In [8]:
def generate_protein_sequences(
    prompts,
    model,
    tokenizer,
    max_new_tokens=170,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
    num_return_sequences=10,
    device="cuda"
):
    """
    prompts: list[str]  → batch prompts
    returns: list[str]  → generated protein sequences
    """

    model.eval()
    model.to(device)

    if isinstance(prompts, str):
        prompts = [prompts]

    # batch tokenize
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=repetition_penalty,
            num_return_sequences=num_return_sequences,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # remove spaces for protein tokens
    decoded = [seq.replace(" ", "") for seq in decoded]

    return decoded

In [9]:
prompt = "[BOS]"
sequences = []
for _ in range(10):
    generated = generate_protein_sequences(prompt,model = model, tokenizer =tokenizer )
    sequences.extend(generated)
    # print(f'Length: {len(generated)}',"Generated Protein Sequence:\n", generated)

In [10]:
import pandas as pd
df = pd.DataFrame(sequences,columns = ['sequence'])
df

,sequence
0,QVQLVQSGSELKKPGASVKVSCRTSAFSSTSYVINWVRQAPGQGLE...
1,GGSLRLSCAASGFTFSSYEMNWVRQAPGKGLEWVSYIGSSSSAIYY...
2,GGSLRLSCAASGFSFSDSYMRWVRQAPGKGLEWVSYISTAGSTIYY...
3,QVQLQESGPGLVKPSETLSLTCAVSGDSVNNTKWWSWVRQPPGKGL...
4,PSETLSLTCTVSGGSISTYHWGWIRQPPGKGLEWVAVIWNSGNSEY...
...,...
95,ASVKVSCKASGYTFSSYDINWVRQATGQGLEWMGWMNPNSGNTGYA...
96,QVQLVQSGAEVKKPGASVKVSCKASGYTFTTYEINWVRQAPGHGLE...
97,QVQLVESGGGVVQPGTSLRLSCAASGFTFTSYDMHWVRQATGKGLE...
98,QITLKESGPTLVKPTQTLTLTCTFSGFSLNTSGVGVGWIRQPPGQA...


In [13]:
df.to_csv(os.path.basename(checkpoint_path)+'.csv',index = False)

In [15]:
import Levenshtein
from itertools import combinations

def novelty_score(generated, training_set):
    training_set = set(training_set)
    novel = [seq for seq in generated if seq not in training_set]
    return len(novel) / len(generated), novel

def diversity_score(sequences):
    """
    Calculates diversity using the formula:
    (2 / (n * (n - 1))) * sum of distances between all pairs.
    Distance = normalized Levenshtein.
    """
    n = len(sequences)
    if n < 2:
        return 0.0  # not enough sequences to compute diversity

    total_dist = 0.0
    for x, y in combinations(sequences, 2):
        dist = Levenshtein.distance(x, y)
        norm_dist = dist / max(len(x), len(y))  # normalize to [0, 1]
        total_dist += norm_dist

    diversity = (2 / (n * (n - 1))) * total_dist
    return diversity

def jaccard_similarity(seq1, seq2, k=3):
    set1 = set(seq1[i:i+k] for i in range(len(seq1)-k+1))
    set2 = set(seq2[i:i+k] for i in range(len(seq2)-k+1))
    return len(set1 & set2) / len(set1 | set2)

# Compare all generated sequences using Jaccard
def average_jaccard(sequences, k=3):
    scores = []
    for a, b in combinations(sequences, 2):
        scores.append(jaccard_similarity(a, b, k))
    return sum(scores) / len(scores) if scores else 1.0

def unique_score(sequences):
    unique = set(sequences)
    return len(unique) / len(sequences),len(sequences)- len(unique)

In [16]:
# Example usage
train_sequences = ds['train'].to_pandas()['sequence'].tolist()  # list of known/training sequences
generated_sequences = list(set(df['sequence'].tolist()))
novelty, novel_seq = novelty_score(generated_sequences, train_sequences)
print(f"Novelty: {novelty:.3f}")

Novelty: 1.000


In [17]:
uni_score, no_of_duplicates = unique_score(df['sequence'].tolist())
print(f"Unique: {uni_score:.3f}")
print('duplicates: ',no_of_duplicates)

Unique: 1.000
duplicates:  0


In [18]:
dev_score = diversity_score(generated_sequences)
print(f"Diversity: {dev_score:.3f}")

Diversity: 0.533


In [19]:
jaccard_score = average_jaccard(generated_sequences)
print(f"Average Jaccard similarity (3-mers): {jaccard_score:.3f}")


Average Jaccard similarity (3-mers): 0.126


# 1000 sample

In [20]:
prompt = "[BOS]"
sequences = []
for _ in range(100):
    generated = generate_protein_sequences(prompt,model = model, tokenizer =tokenizer )
    sequences.extend(generated)
    # print(f'Length: {len(generated)}',"Generated Protein Sequence:\n", generated)

In [21]:
import pandas as pd
df = pd.DataFrame(sequences,columns = ['sequence'])
df

,sequence
0,EVQLVESGGGLVQPGGSLRLSCAASGFTFSRFWMSWVRQAPGKGLE...
1,GESLKISCAVSGFTFKNYGMDWVRQAPGKGLEWVAFINLDGSKIQY...
2,ASVKVSCKASGYTFIEYYLHWVRQAPGQGFEWMGVITPSGGATNYP...
3,GGSLRLSCAASGFTLAKYHMNWVRQAPGKGLEWLSDMFHARSYKDY...
4,QVQLQESGPGLVKPSETLSLTCAVSGYSISSAYYWGWVRQPPGKGL...
...,...
995,QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLE...
996,QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLE...
997,ALVKPTQTLTLTCTVSGFSLNDRGMHWVRQAPGQGLEWLARIDSDE...
998,GESLKISXAASGFTVSSNYMSWVRQAPGKGLEWVSVIYSGGSTYYA...


In [23]:
df.to_csv(os.path.basename(checkpoint_path)+'-1000.csv',index = False)

In [24]:
# Example usage
train_sequences = ds['train'].to_pandas()['sequence'].tolist()  # list of known/training sequences
generated_sequences = list(set(df['sequence'].tolist()))
novelty, novel_seq = novelty_score(generated_sequences, train_sequences)
print(f"Novelty: {novelty:.3f}")


Novelty: 1.000


In [25]:
uni_score, no_of_duplicates = unique_score(df['sequence'].tolist())
print(f"Unique: {uni_score:.3f}")
print('duplicates: ',no_of_duplicates)

Unique: 1.000
duplicates:  0


In [26]:
dev_score = diversity_score(generated_sequences)
print(f"Diversity: {dev_score:.3f}")

Diversity: 0.526


In [27]:
jaccard_score = average_jaccard(generated_sequences)
print(f"Average Jaccard similarity (3-mers): {jaccard_score:.3f}")


Average Jaccard similarity (3-mers): 0.132


In [28]:
import torch
import math
import torch.nn.functional as F

def calculate_perplexity(
    sequences,
    model,
    tokenizer,
    device="cuda",
    batch_size=8
):
    """
    sequences: list[str]  → protein sequences (validation set)
    returns: float → perplexity
    """

    model.eval()
    model.to(device)

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i+batch_size]

            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(device)

            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

            logits = outputs.logits

            # Shift for next-token prediction
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=tokenizer.pad_token_id,
                reduction="sum"
            )

            total_loss += loss.item()
            total_tokens += (shift_labels != tokenizer.pad_token_id).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return perplexity

ppl = calculate_perplexity(
    generated_sequences,
    model,
    tokenizer
)

print("Perplexity:", round(ppl,3))

Perplexity: 5.921
